# QSV2G Stage 06 — Malicious-authority security, QSS/classical comparison, retry model, and formal-candidate check

**Source is pinned to a specific Git commit.** This is a software-only research evaluation. P0/P1 are real ML-KEM-768/ML-DSA-65 probes; P2 is an analytical/Monte Carlo retry model; P3 is a *new unverified* Tamarin candidate. Tamarin outputs must not be called proofs unless its own named lemmas are verified.

Run in a Kaggle **CPU** notebook with **Internet** enabled. The experiment doesn't modify Stage 05. Default below is full mode. Download the exported ZIP and share it for audit.

**Attack scope:** Some P0 positive cases require two compromised authority private keys **and privileged internal station-state control**. They do NOT show a network-only attacker can forge QSS authorization.

## 0. Experiment configuration

In [ ]:
import os,sys,platform,subprocess,shutil,json,csv,hashlib,zipfile,datetime,importlib.util
from pathlib import Path
SMOKE=False  # False=all 96 P0/P1 cases and full P2 modeled grid; True=quick preflight
SEED=2026
REPO_URL='https://github.com/shanmukhdatta/QSV2G.git'
PINNED_SOURCE='857306e6d6fb4d203ba037b9ec671c230c0e703f'
WORK=Path('/kaggle/working' if Path('/kaggle/working').exists() else '/mnt/data')
ROOT=WORK/'QSV2G_stage06_source'
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=WORK/f'qsv2g_stage06_{"smoke" if SMOKE else "full"}_{STAMP}'
OUT.mkdir(parents=True,exist_ok=True)
print('Mode', 'SMOKE' if SMOKE else 'FULL','Output directory',OUT)


## 1. Check out exact source and dependencies

In [ ]:
if not ROOT.exists():
    subprocess.run(['git','clone','--quiet',REPO_URL,str(ROOT)],check=True)
subprocess.run(['git','fetch','--quiet','origin'],cwd=ROOT,check=True)
subprocess.run(['git','checkout','--detach',PINNED_SOURCE],cwd=ROOT,check=True)
actual=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
assert actual==PINNED_SOURCE, (actual,PINNED_SOURCE)
missing=[p for p in ('numpy','pandas','pytest','cryptography') if importlib.util.find_spec(p) is None]
if missing: subprocess.run([sys.executable,'-m','pip','install','--quiet',*missing],check=True)
sys.path.insert(0,str(ROOT/'src'))
from qsv2g_core import PQCrypto
try:
    CRYPTO=PQCrypto()
except RuntimeError:
    subprocess.run([sys.executable,'-m','pip','install','--quiet','pqcrypto>=0.4.0'],check=True)
    CRYPTO=PQCrypto()
assert 'ML-KEM-768' in CRYPTO.kind and 'ML-DSA-65' in CRYPTO.kind
print('Pinned source:',actual,'Crypto backend:',CRYPTO.kind)


## 2. Run dedicated Stage 06 regression tests

In [ ]:
tests=subprocess.run([sys.executable,'-m','pytest','-q','tests/test_stage06.py'],
    cwd=ROOT,capture_output=True,text=True)
(OUT/'stage06_pytest.log').write_text(tests.stdout+'\n'+tests.stderr)
print(tests.stdout[-2500:]);print(tests.stderr[-500:])
if tests.returncode: raise RuntimeError('Stage 06 regression tests failed; inspect stage06_pytest.log')


## 3. P0 + P1: real-PQC adversarial tests and computational-basis parity

In [ ]:
p0p1=OUT/'p0_p1';p0p1.mkdir(exist_ok=True)
cmd=[sys.executable,'experiments/stage06/security.py','--out',str(p0p1),
     '--seed',str(SEED),'--L','9']
if SMOKE:cmd.append('--smoke')
p=subprocess.run(cmd,cwd=ROOT,capture_output=True,text=True)
(p0p1/'process.log').write_text(p.stdout+'\n'+p.stderr)
print(p.stdout[-3000:]);print(p.stderr[-1000:])
if p.returncode:raise RuntimeError('P0/P1 failed: inspect p0_p1/process.log')
import pandas as pd
attacks=pd.read_csv(p0p1/'p0_p1_attack_matrix.csv')
exact=pd.read_csv(p0p1/'p1_exact_distributions.csv')
assert attacks['match'].astype(str).str.lower().eq('true').all()
assert len(attacks)==(16 if SMOKE else 96)
assert len(exact)==9
print(pd.crosstab(attacks.scenario,attacks.outcome).to_string())
print('Maximum classical/QSS computational-basis TV:',exact.computational_basis_total_variation.max())


## 4. P2: fresh-resource retry availability and cost (analytical model)

In [ ]:
p2=OUT/'p2';p2.mkdir(exist_ok=True)
cmd=[sys.executable,'experiments/stage06/reliability.py','--out',str(p2),'--seed',str(SEED)]
if SMOKE:cmd.append('--smoke')
p=subprocess.run(cmd,cwd=ROOT,capture_output=True,text=True)
(p2/'process.log').write_text(p.stdout+'\n'+p.stderr)
print(p.stdout[-1000:]);print(p.stderr[-1000:])
if p.returncode:raise RuntimeError('P2 failed: inspect p2/process.log')
model=pd.read_csv(p2/'p2_retries.csv')
assert len(model)==(80 if SMOKE else 400)
selected=model[(model.L==162)&(model.channel=='x')&(model.p==0.01)].sort_values('retry_budget')
print(selected[['retry_budget','analytic_budget_acceptance','observed_budget_acceptance','analytic_mean_attempts']].to_string(index=False))


## 5. Export a checkpoint before experimental formal verification

In [ ]:
def export_snapshot(stage):
    files=sorted(p for p in OUT.rglob('*') if p.is_file() and not p.name.endswith('.zip') and p.name!='file_hashes.json')
    digest={str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in files}
    (OUT/'file_hashes.json').write_text(json.dumps(digest,indent=2))
    archive=WORK/f'qsv2g_stage06_{"smoke" if SMOKE else "full"}_{STAMP}_{stage}.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in files+[OUT/'file_hashes.json']:
            z.write(p,str(p.relative_to(OUT)))
    print('Snapshot:',archive,archive.stat().st_size,'bytes')
    return archive
checkpoint=export_snapshot('checkpoint_before_P3')


## 6. P3: attempt the **candidate** Tamarin model (not guaranteed)

The candidate is **not yet verified**. The notebook may install pinned official Tamarin 1.12.0 and Maude 3.5.1 releases using the existing Stage 05 verified-digest installer. A successful installation does **not** prove the model. A parser error, falsified lemma, or timeout is a research finding and must be reported honestly.

In [ ]:
p3=OUT/'p3';p3.mkdir(exist_ok=True)
TOOL_DIR=WORK/'qsv2g_stage06_tamarin_tools'
TAM_BIN=TOOL_DIR/'bin/tamarin-prover'
install_status=TOOL_DIR/'install_status.json'
if not TAM_BIN.exists():
    install_cmd=[sys.executable,'experiments/stage05/install_tamarin.py',
        '--tools-dir',str(TOOL_DIR),'--status',str(install_status)]
    proc=subprocess.run(install_cmd,cwd=ROOT,capture_output=True,text=True)
    (p3/'installation.log').write_text(proc.stdout+'\n'+proc.stderr)
else:
    (p3/'installation.log').write_text('Previously installed binary reused from '+str(TAM_BIN))
if install_status.exists():
    shutil.copy2(install_status,p3/'install_status.json')
# Subprocess wrapper records NOT_EXECUTED/UNRESOLVED rather than inventing a result.
cmd=[sys.executable,'experiments/stage06/run_formal.py','--out',str(p3),
     '--timeout','360','--tamarin-bin',str(TAM_BIN)]
proc=subprocess.run(cmd,cwd=ROOT,capture_output=True,text=True)
(p3/'runner.log').write_text(proc.stdout+'\n'+proc.stderr)
print(proc.stdout[-2000:]); print(proc.stderr[-1000:])
proof=json.loads((p3/'p3_status.json').read_text()) if (p3/'p3_status.json').exists() else {'status':'RUNNER_ERROR'}
print('P3:',proof.get('status'),'— P3 evidence is conditional on actual prover logs.')


## 7. Final summary, archive and provenance

In [ ]:
summary={'stage':'06','mode':'SMOKE' if SMOKE else 'FULL',
 'git_sha':actual,'crypto_backend':CRYPTO.kind,'python':sys.version,
 'platform':platform.platform(),
 'p0_p1_attack_cases':len(attacks),
 'p0_p1_all_expected':bool(attacks['match'].astype(str).str.lower().eq('true').all()),
 'conditional_no_reconstruction_acceptances':int((attacks.outcome=='EV_ACCEPTED_NO_RECONSTRUCTION').sum()),
 'exact_qss_classical_basis_pairs':len(exact),
 'exact_qss_classical_basis_TV_max':float(exact.computational_basis_total_variation.max()),
 'p2_model_configs':len(model),
 'p3_tamarin_status':proof.get('status','UNKNOWN'),
 'p3_proof_status':proof.get('proof_status','UNRESOLVED'),
 'scope':['No network-only intrusion proof','No real quantum channel',
          'QSS state not maliciously secure VQSS',
          'P2 only a hypothetical retry model',
          'P3 only an abstract candidate']}
(OUT/'stage06_run_summary.json').write_text(json.dumps(summary,indent=2))
final_zip=export_snapshot('results')
print(json.dumps(summary,indent=2))
print('Upload this ZIP for independent audit:',final_zip)
